In [3]:
import os
import re
import string
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, SpatialDropout1D, BatchNormalization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

if not os.path.exists('train.csv') or not os.path.exists('test.csv'):
    !wget -q https://raw.githubusercontent.com/LawrenceDuan/IMDb-Review-Analysis/master/IMDb_Reviews.csv -O full_imdb.csv
    full_df = pd.read_csv('full_imdb.csv')
    full_df.columns = ['text', 'label']
    tr_df, te_df = train_test_split(full_df, test_size=0.5, random_state=42, stratify=full_df['label'])
    tr_df.to_csv('train.csv', index=False)
    te_df.to_csv('test.csv', index=False)

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

def clean_review(text):
    text = str(text).lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    return text.translate(str.maketrans('', '', string.punctuation))

train_data['text'] = train_data['text'].apply(clean_review)
test_data['text'] = test_data['text'].apply(clean_review)

if train_data['label'].dtype == 'object':
    mapping = {'positive': 1, 'negative': 0, '1': 1, '0': 0}
    train_data['label'] = train_data['label'].map(mapping)
    test_data['label'] = test_data['label'].map(mapping)
else:
    train_data['label'] = train_data['label'].astype(int)
    test_data['label'] = test_data['label'].astype(int)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    train_data['text'], train_data['label'],
    test_size=0.15, random_state=42, stratify=train_data['label']
)

X_test_raw = test_data['text']
y_test = test_data['label']

VOCAB_SIZE = 15000
MAX_LEN = 250
EMBEDDING_DIM = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<UNK>")
tokenizer.fit_on_texts(X_train_raw)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train_raw), maxlen=MAX_LEN, padding='pre', truncating='pre')
X_val = pad_sequences(tokenizer.texts_to_sequences(X_val_raw), maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=MAX_LEN, padding='pre', truncating='pre')

y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

if not os.path.exists('glove.6B.100d.txt'):
    if not os.path.exists('glove.6B.zip'):
        !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q -o glove.6B.zip glove.6B.100d.txt

embeddings_index = {}
all_vectors = []
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
        all_vectors.append(coefs)

glove_mean = np.mean(all_vectors, axis=0)

glove_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))
glove_matrix[0] = np.zeros(EMBEDDING_DIM)
glove_matrix[1] = glove_mean

for word, i in tokenizer.word_index.items():
    if 1 < i < VOCAB_SIZE:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            glove_matrix[i] = embedding_vector
        else:
            glove_matrix[i] = glove_mean

shared_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=1)
]

model_A_scratch = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    SpatialDropout1D(0.3),
    GRU(128, return_sequences=True),
    Dropout(0.3),
    GRU(64, return_sequences=False),
    Dropout(0.4),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_A_scratch.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model_A_scratch.fit(X_train, y_train, epochs=6, batch_size=64, validation_data=(X_val, y_val), callbacks=shared_callbacks)

model_A_glove = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM,
              embeddings_initializer=tf.keras.initializers.Constant(glove_matrix), trainable=False),
    SpatialDropout1D(0.3),
    GRU(128, return_sequences=True),
    Dropout(0.3),
    GRU(64, return_sequences=False),
    Dropout(0.4),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_A_glove.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model_A_glove.fit(X_train, y_train, epochs=4, batch_size=64, validation_data=(X_val, y_val))

model_A_glove.layers[0].trainable = True
model_A_glove.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5), loss='binary_crossentropy', metrics=['accuracy'])
model_A_glove.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_val, y_val), callbacks=shared_callbacks)

loss_As, acc_As = model_A_scratch.evaluate(X_test, y_test, verbose=0)
loss_Ag, acc_Ag = model_A_glove.evaluate(X_test, y_test, verbose=0)

print(f"Scratch: {loss_As:.4f} / {acc_As*100:.2f}%")
print(f"GloVe: {loss_Ag:.4f} / {acc_Ag*100:.2f}%")

Epoch 1/6
333/333 ━━━━━━━━━━━━━━━━━━━━ 13s 29ms/step - accuracy: 0.7182 - loss: 0.5307 - val_accuracy: 0.7976 - val_loss: 0.4933 - learning_rate: 0.0010
Epoch 2/6
333/333 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.8905 - loss: 0.2719 - val_accuracy: 0.8728 - val_loss: 0.2965 - learning_rate: 0.0010
Epoch 3/6
333/333 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.9357 - loss: 0.1713 - val_accuracy: 0.8683 - val_loss: 0.3606 - learning_rate: 0.0010
Epoch 4/6
333/333 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.9746 - loss: 0.0821 - val_accuracy: 0.8811 - val_loss: 0.3469 - learning_rate: 2.0000e-04
Epoch 5/6
333/333 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.9841 - loss: 0.0568 - val_accuracy: 0.8851 - val_loss: 0.3577 - learning_rate: 4.0000e-05
Epoch 1/4
333/333 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - accuracy: 0.5622 - loss: 0.7402 - val_accuracy: 0.6059 - val_loss: 0.7408
Epoch 2/4
333/333 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.7304 - loss: 0.5408 - val_accuracy: